# Deriv Synthetic Pairs — Strategy Backtest
Runs against **real Deriv demo data** (V10, V25, V50, V100).

Paste your demo API token in the cell below, then **Runtime → Run all**.

In [ ]:
# ── 1. Install deps ──────────────────────────────────────────────
!pip install websockets numpy pandas colorama tabulate -q

In [ ]:
# ── 2. Paste your Deriv DEMO API token here ──────────────────────
API_TOKEN = "pat_0ab2fa965c55bde0a840138bd6aaa1ec5653981f3311f7f75c379865c41d494e"

# Symbols to test
SYMBOLS = {"R_10": "Volatility 10", "R_25": "Volatility 25",
           "R_50": "Volatility 50", "R_100": "Volatility 100"}
TICK_COUNT   = 5000
PAYOUT_RATIO = 0.87
STAKE        = 1.0

In [ ]:
# ── 3. Indicators ────────────────────────────────────────────────
import numpy as np

def rsi(prices, period=14):
    if len(prices) < period + 1: return 50.0
    d = np.diff(prices[-(period+1):])
    g, l = np.where(d>0,d,0).mean(), np.where(d<0,-d,0).mean()
    return 100.0 if l==0 else 100-(100/(1+g/l))

def ema(prices, period):
    if len(prices) < period: return [float('nan')]*len(prices)
    k = 2/(period+1); r = [float('nan')]*(period-1)
    r.append(float(np.mean(prices[:period])))
    for p in prices[period:]: r.append(p*k+r[-1]*(1-k))
    return r

def bollinger(prices, period=20, std=2.0):
    if len(prices)<period: m=float(np.mean(prices)); return m,m,m
    w=prices[-period:]; m=float(np.mean(w)); s=float(np.std(w,ddof=1))
    return m+std*s, m, m-std*s

def atr(prices, period=14):
    if len(prices)<2: return 0.0
    trs=[abs(prices[-i]-prices[-(i+1)]) for i in range(1,min(period+1,len(prices)))]
    return float(np.mean(trs)) if trs else 0.0

print('Indicators OK')

In [ ]:
# ── 4. Strategies ────────────────────────────────────────────────
def sig_rsi(prices, p={"period":14,"oversold":30,"overbought":70,"duration":5}):
    if len(prices)<p["period"]+5: return None
    r=rsi(prices,p["period"])
    if r<p["oversold"]:  return {"ct":"CALL","dur":p["duration"]}
    if r>p["overbought"]: return {"ct":"PUT", "dur":p["duration"]}

def sig_bb(prices, p={"period":20,"std":2.0,"duration":5}):
    if len(prices)<p["period"]+1: return None
    u,m,l=bollinger(prices,p["period"],p["std"]); pr=prices[-1]
    if pr<l: return {"ct":"CALL","dur":p["duration"]}
    if pr>u: return {"ct":"PUT", "dur":p["duration"]}

def sig_ema(prices, p={"fast":5,"slow":20,"duration":3}):
    need=p["slow"]+3
    if len(prices)<need: return None
    f=ema(prices,p["fast"]); s=ema(prices,p["slow"])
    fc,fp,sc,sp=f[-1],f[-2],s[-1],s[-2]
    if any(v!=v for v in [fc,fp,sc,sp]): return None
    if fp<=sp and fc>sc: return {"ct":"CALL","dur":p["duration"]}
    if fp>=sp and fc<sc: return {"ct":"PUT", "dur":p["duration"]}

def sig_momentum(prices, p={"period":10,"mult":1.5,"duration":5}):
    n=p["period"]
    if len(prices)<n+2: return None
    w=prices[-n:]; m=sum(w)/len(w); at=atr(prices,n); band=at*p["mult"]
    pr,pv=prices[-1],prices[-2]
    if pv<=m+band<pr: return {"ct":"CALL","dur":p["duration"]}
    if pv>=m-band>pr: return {"ct":"PUT", "dur":p["duration"]}

STRATEGIES = {
    "RSI Reversal":      sig_rsi,
    "Bollinger Bands":   sig_bb,
    "EMA Crossover":     sig_ema,
    "Momentum Breakout": sig_momentum,
}
print('Strategies OK')

In [ ]:
# ── 5. Fetch real historical ticks from Deriv ────────────────────
import asyncio, json, websockets

WS_URL = "wss://ws.binaryws.com/websockets/v3?app_id=1089"

async def fetch_ticks(token, symbol, count=5000):
    async with websockets.connect(WS_URL) as ws:
        # authorize
        await ws.send(json.dumps({"authorize": token, "req_id": 1}))
        while True:
            msg = json.loads(await ws.recv())
            if msg.get("msg_type") == "authorize":
                if msg.get("error"): raise RuntimeError(msg["error"]["message"])
                break
        # fetch history
        await ws.send(json.dumps({
            "ticks_history": symbol, "count": count,
            "end": "latest", "style": "ticks", "req_id": 2
        }))
        while True:
            msg = json.loads(await ws.recv())
            if msg.get("req_id") == 2:
                if msg.get("error"): raise RuntimeError(msg["error"]["message"])
                return [float(p) for p in msg["history"]["prices"]]

async def get_account(token):
    async with websockets.connect(WS_URL) as ws:
        await ws.send(json.dumps({"authorize": token, "req_id": 1}))
        while True:
            msg = json.loads(await ws.recv())
            if msg.get("msg_type") == "authorize":
                if msg.get("error"): raise RuntimeError(msg["error"]["message"])
                return msg["authorize"]

# Verify connection
account = await get_account(API_TOKEN)
print(f"Connected! Account: {account.get('loginid')}  Balance: {account.get('balance')} {account.get('currency')}")

In [ ]:
# ── 6. Run backtest on all symbols ───────────────────────────────
from collections import defaultdict

results = defaultdict(dict)   # results[symbol][strategy] = stats dict

for symbol, name in SYMBOLS.items():
    print(f"\nFetching {TICK_COUNT} ticks for {name} ({symbol})...")
    prices = await fetch_ticks(API_TOKEN, symbol, TICK_COUNT)
    print(f"  Got {len(prices)} ticks  |  range: {min(prices):.4f} – {max(prices):.4f}")

    for strat_name, strat_fn in STRATEGIES.items():
        wins=losses=signals=0
        for i in range(60, len(prices)-10):
            sig = strat_fn(prices[:i])
            if sig is None: continue
            signals += 1
            end = min(i + sig["dur"], len(prices)-1)
            win = prices[end] > prices[i] if sig["ct"]=="CALL" else prices[end] < prices[i]
            if win: wins += 1
            else:   losses += 1

        wr  = wins/signals if signals else 0
        net = wins*STAKE*PAYOUT_RATIO - losses*STAKE
        pf  = (wins*PAYOUT_RATIO)/losses if losses else float('inf')
        results[symbol][strat_name] = {"signals":signals,"wins":wins,"losses":losses,
                                        "wr":wr,"net":net,"pf":pf,"score":wr*pf}
        flag = '✅' if net>0 else '❌'
        print(f"  {flag} {strat_name:<22} sig={signals:>4}  WR={wr:.1%}  PnL={net:+.2f}  PF={pf:.2f}")

print("\nBacktest complete!")

In [ ]:
# ── 7. Final report + recommendations ───────────────────────────
print("\n" + "="*65)
print("  RESULTS SUMMARY")
print("="*65)

recommendations = []
for symbol, name in SYMBOLS.items():
    strats = results[symbol]
    best = max(strats, key=lambda s: strats[s]["score"])
    b = strats[best]
    profitable = [s for s,d in strats.items() if d["net"]>0]
    print(f"\n  {name} ({symbol})")
    for s,d in sorted(strats.items(), key=lambda x: -x[1]["score"]):
        flag = '✅' if d['net']>0 else '❌'
        print(f"    {flag} {s:<22}  WR={d['wr']:.1%}  PnL={d['net']:+.2f}  PF={d['pf']:.2f}")
    print(f"    → Best: {best}  (WR={b['wr']:.1%}, PnL={b['net']:+.2f})")
    recommendations.append((symbol, name, best, b))

print("\n" + "="*65)
print("  BOT DESIGN RECOMMENDATION")
print("="*65)
risk_map = {"R_10":"fixed","R_25":"dalembert","R_50":"anti_martingale","R_100":"fixed"}
dur_map  = {"R_10":"1-3","R_25":"3-5","R_50":"5-10","R_100":"3-5"}
for symbol, name, best, b in recommendations:
    print(f"  {name}")
    print(f"    Strategy : {best}")
    print(f"    Win Rate : {b['wr']:.1%}  |  PnL: {b['net']:+.2f}  |  PF: {b['pf']:.2f}")
    print(f"    Risk Mode: {risk_map[symbol]}")
    print(f"    Duration : {dur_map[symbol]} ticks")
    print()